# Build NRSS morphology from open3d voxel grid and material optical constants

## Setup

### Imports

In [ ]:
# Imports:
import pathlib
import subprocess
import io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
import xarray as xr
import open3d as o3d
import pandas as pd
import dask.array as da
from tqdm.auto import tqdm

from NRSS.writer import write_materials
from NRSS.morphology import Morphology, Material, OpticalConstants

### Define paths

In [ ]:
notebookPath = pathlib.Path.cwd()
zarrPath = notebookPath.joinpath('material_optical_constants.zarr')
zarrPath.exists()

## Load material optical constants, generate CyRSoXS material files

### Load optical constant xarray dataarrays

In [ ]:
oc_DS = xr.open_zarr(zarrPath).sortby('theta')

# Compute any dask coordiantes
for coord_name, coord_data in oc_DS.coords.items():
    if isinstance(coord_data.data, da.Array):
        oc_DS.coords[coord_name] = coord_data.compute()
        
display(oc_DS)

### Generate CyRSoXS material files

In [ ]:
# Extract arrays
Energy = oc_DS['energy'].data
BetaPara = oc_DS.sel(sample_name='PM6_CBCN_rot', theta=0)['beta'].data.compute()
BetaPerp = oc_DS.sel(sample_name='PM6_CBCN_rot', theta=90)['beta'].data.compute()
DeltaPara = oc_DS.sel(sample_name='PM6_CBCN_rot', theta=0)['delta'].data.compute()
DeltaPerp = oc_DS.sel(sample_name='PM6_CBCN_rot', theta=90)['delta'].data.compute()

# Stack into columns
PM6_oc_arr = np.vstack((Energy,BetaPara,BetaPerp,DeltaPara,DeltaPerp)).T

# Check shape
display(PM6_oc_arr.shape)

In [ ]:
# Save as txt file, to current working directory
oc_savename = 'PM6_oc.txt'
np.savetxt(oc_savename, PM6_oc_arr)

In [ ]:
energies = np.hstack((np.arange(270,283, 2), np.arange(283,290,0.2), np.arange(290,320,2)))
display(len(energies))
display(energies)

In [ ]:
# Write to materials files, to current working directory
# energies = np.round(np.arange(275,320,0.1),1)  # Set energies to use
energies = np.hstack((np.arange(270,283, 2), np.arange(283,290,0.2), np.arange(290,320,2)))  # set energies
material_dict = {'Material1':oc_savename}
energy_dict = {'Energy':0, 'BetaPara':1, 'BetaPerp':2, 'DeltaPara':3, 'DeltaPerp':4}  

write_materials(energies, material_dict, energy_dict, 1)

## Load open3d voxel grid .ply, generate NRSS morphology

In [ ]:
# Define simple functions
def to_s0to1(value, min=-180, max=180):
    """Adjust a linear scale from min to max to fit between 0 and 1"""
    shift = 0 - min
    max = max + shift
    return (value + shift) / max

def from_s0to1(value, min=-180, max=180):
    """Inverse of 'to_s0to1': Adjust a linear scale 0 to 1 to an arbitrary linear scale between min and max"""
    shift = 0 + min
    max = max - shift
    return (value * max) + shift

### Load .ply

In [ ]:
# vgridPath = notebookPath.joinpath('open3d_voxel_plys','RBD03_perp2bb_voxel_grid_v1.ply')
# vgridPath = notebookPath.joinpath('open3d_voxel_plys','RBD04_para2bb_fixedfibrils_every1points_v1.ply')
# vgridPath = notebookPath.joinpath('open3d_voxel_plys','RBD04_perp2bb_fixedfibrils_every2points_v1.ply')
# vgridPath = notebookPath.joinpath('open3d_voxel_plys','RBD04_para2bb_fixedfibrils_every2points_v1.ply')
vgridPath = notebookPath.joinpath('open3d_voxel_plys','RBD04_perp2bb_fixedfibrils_every1points_v1.ply')

loaded_voxel_grid = o3d.io.read_voxel_grid(str(vgridPath))
loaded_voxel_grid

In [ ]:
# Convert voxel grid to list of voxels, with grid index & color
voxels = loaded_voxel_grid.get_voxels()  # returns list of voxels

In [ ]:
voxel_indices = np.array(list(map(lambda x: x.grid_index, voxels[:])))
voxel_colors = np.array(list(map(lambda x: x.color, voxels[:])))

In [ ]:
voxel_indices.max()

In [ ]:
voxel_indices[:,2].max()

In [ ]:
too_large = (voxel_indices[:,[0,1]] > 1023).nonzero()[0]
voxel_indices = np.delete(voxel_indices, too_large, axis=0)
voxel_colors = np.delete(voxel_colors, too_large, axis=0)

too_large = (voxel_indices[:,2] > 127).nonzero()[0]
voxel_indices = np.delete(voxel_indices, too_large, axis=0)
voxel_colors = np.delete(voxel_colors, too_large, axis=0)

display(voxel_indices.shape)
display(voxel_colors.shape)

### Generate NRSS morpohlogy 
With material optical constants above

In [ ]:
mat1_Vfrac = np.zeros((128,1024,1024))
mat1_psi = np.zeros((128,1024,1024))
mat1_theta = np.zeros((128,1024,1024))
mat1_S = np.zeros((128,1024,1024))

In [ ]:
# Assign volume fraction of one for all voxel positions
# mat1_Vfrac[voxel_indices[:,2],voxel_indices[:,1],voxel_indices[:,0]] = 1.
mat1_Vfrac = mat1_Vfrac + 1
mat1_psi[voxel_indices[:,2],voxel_indices[:,1],voxel_indices[:,0]] = np.deg2rad(from_s0to1(voxel_colors[:,0], min=-180, max=180))
mat1_theta[voxel_indices[:,2],voxel_indices[:,1],voxel_indices[:,0]] = np.deg2rad(from_s0to1(voxel_colors[:,1], min=0, max=90))
mat1_S[voxel_indices[:,2],voxel_indices[:,1],voxel_indices[:,0]] = voxel_colors[:,2]

In [ ]:
mat2_Vfrac = mat2_psi = mat2_theta = mat2_S = np.zeros((128,1024,1024))

In [ ]:
# load optical constants from previous Material1.txt file
mat1_consts = OpticalConstants.load_matfile(notebookPath.joinpath('Material1.txt'),name='Material 1')

#create Material objects to hold the relevant voxel and optical constant information
mat1 = Material(materialID=1,Vfrac=mat1_Vfrac,S=mat1_S,
                psi=mat1_psi,theta=mat1_theta,energies=energies,
                opt_constants=mat1_consts.opt_constants,name='Material 1')

#automatically assigns zeros for optical constants if the name is vacuum
mat2 = Material(materialID=2,Vfrac=mat2_Vfrac,S=mat2_S,
                psi=mat2_psi,theta=mat2_theta,energies=energies,name='vacuum')

In [ ]:
morph1 = Morphology(2,materials={1:mat1,2:mat2},PhysSize=2.15, create_cy_object=True)

In [ ]:
morph1.inputData.print()

In [ ]:
morph1.create_update_cy()

In [ ]:
morph1.check_materials(quiet=False)

In [ ]:
morph1.visualize_materials(z_slice=64)

In [ ]:
# %matplotlib widget
# plt.close('all')

In [ ]:
# #demonstrating some of the options on visualize_materials, including how to use its ability to return and redisplay RGBA arrays containing faithful visualization
# plt_img = morph1.visualize_materials(
#     z_slice=32,
#     subsample=64, #makes the visualized area 20x20
#     translate_x = +0, #moves the visualized area 6 voxels right from center; intended to be used with subsample
#     translate_y = +0, #moves the visualized area 6 voxels up from center; intended to be used with subsample
#     plotstyle="dark", #makes the plot background dark and the annotations white
#     add_quiver=True, #adds lines to psi plot indicating in-plane orientation of Euler angles
#     outputaxes = False, #suppresses output of axes labels and colorbar
#     outputmat=[1], #selects material 1 to return visualization
#     outputplot=["psi"], #selects psi map to return visualization
#     runquiet=True, #do not create full display of all materials; intended to be used with outputmat & outputplot
# )[0] #return will be a list (multiple plots can be returned) so be sure to select an item on the list 
# plt.style.use("dark_background")
# plt.figure(figsize=(10, 10))
# plt.imshow(plt_img) # the visualization created by visualize_materials can be redisplayed using imshow
# plt.axis("off") # in general you will not want axes on this redisplay plot because they will describe the visualize_materials image and no the data
# plt.show()

## Simulate with CyRSoXS!

In [ ]:
morph1.EAngleRotation = [0, 1, 360]
name='0-1-360_EAngleRot'

In [ ]:
scattering_data = morph1.run()

In [ ]:
extent = 1
sliced_DA = scattering_data.sel(qx=slice(-extent, extent), qy=slice(-extent,extent))
# cmin, cmax = sliced_DA.sel(energy=285,method='nearest').quantile([0.001, 0.99]).data

for energy in sliced_DA.energy.data:
    cmin, cmax = sliced_DA.sel(energy=energy,method='nearest').quantile([0.001, 0.99]).data
    sliced_DA.sel(energy=energy, method='nearest').plot.imshow(norm=LogNorm(cmin,cmax), cmap=plt.cm.turbo)
    plt.show()

In [ ]:
# name='1-EAngleRot'
cyrsoxs_result_DS = scattering_data.to_dataset(name=name)
# cyrsoxs_result_DS.to_zarr(notebookPath.joinpath('cyrsoxs_outputs', 'RBD04_para2bb_fixedfibrils_every1points_v1', f'{name}.zarr'))
# cyrsoxs_result_DS.to_zarr(notebookPath.joinpath('cyrsoxs_outputs', 'RBD04_perp2bb_fixedfibrils_every2points_v1', f'{name}.zarr'))
# cyrsoxs_result_DS.to_zarr(notebookPath.joinpath('cyrsoxs_outputs', 'RBD04_para2bb_fixedfibrils_every2points_v1', f'{name}.zarr'))
cyrsoxs_result_DS.to_zarr(notebookPath.joinpath('cyrsoxs_outputs', 'RBD04_perp2bb_fixedfibrils_every1points_v1', f'{name}.zarr'))